# Kaggle submission — Gemma 2 9B, checkpoint-1440

**Competition:** LLM Classification Finetuning (`llm-classification-finetuning`).

1. Run `4_export_9b_checkpoint1440.ipynb` locally in a separate kernel, without interrupting training.
2. Upload its `arena_9b_checkpoint1440_assets.zip` as a **new private Kaggle Dataset**.
3. Import this notebook. **Add Input:** attach that dataset and the competition data.
4. Select **GPU T4 ×2**, set **Internet OFF**, then **Save Version → Save & Run All**.
5. When the saved version succeeds, submit its **`submission.csv`** from the competition submission panel.

**This submission uses only checkpoint-1440**, original A/B order, temperature 1, and no ensemble.
It does not wait for final training/evaluation files and never chooses a later checkpoint automatically.

### Precision and memory
On two T4s, the backbone uses **FP16**, split across the two GPUs. The saved adapter/head parameters are kept FP32;
autocast performs their matrix operations in the selected 16-bit compute dtype. No 4/8-bit quantization is used.
On a sufficiently large native-BF16 GPU, BF16 is used instead. Eager attention and softcapping are retained.

FP16 is a deployment change from BF16 training, not a promise of identical probabilities. The notebook validates
**all saved selection IDs** against checkpoint-1440's recorded log loss before making a submission file. This is a
compatibility check, not a new independent validation result. A material mismatch or non-finite result stops execution.
No arbitrary predictions are substituted to hide errors.

**Keep this notebook separate from your local training kernel.** Importing code does not upload model weights.

In [ ]:
import json
import os
import sys
import time
import shutil
import zipfile
from pathlib import Path

NOTEBOOK_STARTED = time.perf_counter()
ASSET_DIR = None  # Auto-detect the checkpoint-1440 private dataset.
COMPETITION_DIR = None
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 2
MAX_RELOAD_LOGLOSS_DIFFERENCE = 0.01  # Diagnostic guard, not a hyperparameter fitted to test labels.

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


if ASSET_DIR is None:
    matches = []
    for p in INPUT_ROOT.rglob("checkpoint_manifest.json"):
        candidate = read_json(p)
        if candidate.get("bundle_format") == "arena-gemma2-9b-checkpoint-v1" and candidate.get("checkpoint_step") == 1440:
            matches.append(p)
    if not matches:
        archives = list(INPUT_ROOT.rglob("arena_9b_checkpoint1440_assets.zip"))
        if len(archives) == 1:
            extract_dir = WORK_DIR / "_checkpoint1440_assets"
            extract_dir.mkdir(exist_ok=True)
            with zipfile.ZipFile(archives[0]) as z:
                required = sum(m.file_size for m in z.infolist())
                if shutil.disk_usage(WORK_DIR).free < required + 1024**3:
                    raise RuntimeError("Insufficient disk to extract ZIP. Attach the dataset with files expanded.")
                for member in z.infolist():
                    target = (extract_dir / member.filename).resolve()
                    if not target.is_relative_to(extract_dir.resolve()):
                        raise ValueError("Unsafe ZIP path.")
                z.extractall(extract_dir)
            matches = list(extract_dir.rglob("checkpoint_manifest.json"))
    if len(matches) != 1:
        raise FileNotFoundError("Attach exactly one checkpoint-1440 bundle, or set ASSET_DIR explicitly.")
    ASSET_DIR = matches[0].parent
else:
    ASSET_DIR = Path(ASSET_DIR)
manifest = read_json(ASSET_DIR / "checkpoint_manifest.json")
assert manifest["bundle_format"] == "arena-gemma2-9b-checkpoint-v1"
assert manifest["checkpoint_step"] == 1440 and manifest["model_name"] == "google/gemma-2-9b-it"
assert manifest["competition"] == "llm-classification-finetuning"
assert manifest["inference_recipe"] == {"swap_tta": False, "temperature": 1.0, "ensemble": False}
BASE_DIR = ASSET_DIR / manifest["base_subdir"]
ADAPTER_DIR = ASSET_DIR / manifest["adapter_subdir"]
preprocessing = read_json(ASSET_DIR / "preprocessing.json")
print("Using:", ASSET_DIR)
print("Checkpoint:", manifest["checkpoint_step"], "| recorded selection log loss:", manifest["checkpoint_selection_log_loss"])

## 1. Install bundled dependencies offline

Do this before importing Transformers/PEFT. PyTorch/CUDA, NumPy, and pandas are not replaced.
Use a fresh Kaggle session after modifying the attached dataset or installing other model libraries.

In [ ]:
import importlib
import subprocess
from importlib import metadata

if any(name in sys.modules for name in ["transformers", "peft", "accelerate", "tokenizers"]):
    raise RuntimeError("Model libraries are already imported. Restart this Kaggle session and run from the top.")
py = f"{sys.version_info.major}.{sys.version_info.minor}"
if py not in manifest["runtimes"]:
    raise RuntimeError(f"Bundle supports Python {list(manifest['runtimes'])}, but Kaggle uses {py}. Re-export with that version.")
runtime = manifest["runtimes"][py]
wheel_dir = ASSET_DIR / runtime["wheel_subdir"]
runtime_dir = WORK_DIR / "_arena_9b_runtime"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--no-index", "--find-links", str(wheel_dir), "--no-deps", "--target", str(runtime_dir),
    "--upgrade", "-r", str(wheel_dir / "requirements.txt")])
sys.path.insert(0, str(runtime_dir))
importlib.invalidate_caches()
from packaging.requirements import Requirement
for requirement in runtime["preserved_requirements"]:
    req = Requirement(requirement)
    if not req.specifier.contains(metadata.version(req.name), prereleases=True):
        raise RuntimeError(f"Kaggle's {req.name}=={metadata.version(req.name)} does not satisfy {requirement}.")
print("Offline runtime installed. Kaggle's existing CUDA/PyTorch was preserved.")

## 2. Verify assets and read the competition CSVs

All test rows must come from the **competition input**, not the private asset dataset.
The three example test rows are replaced by the hidden test when Kaggle scores the notebook.
The base and adapter hashes detect incomplete or incorrect uploads before model loading.

In [ ]:
import gc
import hashlib
import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from safetensors import safe_open
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from tqdm.auto import tqdm

# Must happen BEFORE PEFT dispatches adapter modules. This model does not use TorchAO.
import peft.tuners.lora.torchao as peft_torchao
peft_torchao.is_torchao_available = lambda: False


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


for rel, info in tqdm(manifest["files"].items(), desc="Verify uploaded assets"):
    p = ASSET_DIR / rel
    assert p.is_file() and p.stat().st_size == info["size"], f"Missing/incomplete file: {rel}"
    assert sha256_file(p) == info["sha256"], f"Checksum mismatch: {rel}"
assert manifest["files"]["adapter/adapter_model.safetensors"]["sha256"] == manifest["adapter_sha256"]

if COMPETITION_DIR is None:
    default_dir = INPUT_ROOT / "llm-classification-finetuning"
    if (default_dir / "test.csv").is_file():
        COMPETITION_DIR = default_dir
    else:
        candidates = [p.parent for p in INPUT_ROOT.rglob("test.csv")
                      if "llm-classification-finetuning" in p.parts]
        if len(candidates) != 1:
            raise FileNotFoundError("Use Add Input to attach LLM Classification Finetuning competition data.")
        COMPETITION_DIR = candidates[0]
else:
    COMPETITION_DIR = Path(COMPETITION_DIR)
LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
LABEL_NAMES = ["win_a", "win_b", "tie"]
assert manifest["label_columns"] == LABEL_COLUMNS and manifest["label_names"] == LABEL_NAMES

test_frame = pd.read_csv(COMPETITION_DIR / "test.csv", dtype={"id": str})
assert len(test_frame) > 0 and test_frame["id"].is_unique
assert {"id", "prompt", "response_a", "response_b"}.issubset(test_frame.columns)
selection_ids = pd.read_csv(ASSET_DIR / "selection_ids.csv", dtype={"id": str})["id"].tolist()
assert len(selection_ids) == manifest["selection_rows"] and len(set(selection_ids)) == len(selection_ids)
validation_raw = pd.read_csv(COMPETITION_DIR / "train.csv", dtype={"id": str})
assert validation_raw["id"].is_unique and set(selection_ids).issubset(validation_raw["id"])
selection_frame = validation_raw.set_index("id").loc[selection_ids].reset_index()
del validation_raw
one_hot = selection_frame[LABEL_COLUMNS].to_numpy()
assert np.isin(one_hot, [0, 1]).all() and (one_hot.sum(1) == 1).all()
y_selection = one_hot.argmax(1).astype(np.int64)

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), local_files_only=True)
tokenizer.padding_side = "right"
assert all(x is not None for x in [tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id])
assert preprocessing["version"] == "full-if-fits-v2"
MAX_LENGTH, PROMPT_BUDGET = preprocessing["max_length"], preprocessing["prompt_budget"]
assert (MAX_LENGTH, PROMPT_BUDGET) == (2048, 384)
assert preprocessing["padding_side"] == "right" and preprocessing["label_names"] == LABEL_NAMES
prefix_ids, a_header_ids, b_header_ids, suffix_ids, omission_ids = [
    tokenizer.encode(preprocessing[key], add_special_tokens=False)
    for key in ["prefix", "a_header", "b_header", "suffix", "omission"]
]
overhead = sum(map(len, [prefix_ids, a_header_ids, b_header_ids, suffix_ids])) + 2
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
print(f"Test rows: {len(test_frame):,} | selection rows to verify: {len(selection_frame):,}")
print({n:metadata.version(n) for n in ["torch", "transformers", "peft", "accelerate"]})

## 3. Exact 9B training preprocessing

The five functions below are copied from the uploaded 9B notebook. They preserve full text when it fits,
otherwise allocate section budgets and retain head/tail content. Do not replace this with the old 2B tokenizer function.
Training-time random swaps are **not** applied here: predictions use the original input order and A/B/tie columns.

In [ ]:
def parse_turns(value):
    turns = json.loads(value) if isinstance(value, str) else value
    if turns is None:
        return [""]
    if not isinstance(turns, list) or not all(t is None or isinstance(t, str) for t in turns):
        raise ValueError("Each competition text field must decode to a list of strings/nulls.")
    return ["" if t is None else t for t in turns] or [""]


def format_turns(turns):
    return "\n\n".join(f"[Turn {i + 1}]\n{t if t else '[No text]'}" for i, t in enumerate(turns))


def trim_tokens(ids, budget):
    if len(ids) <= budget:
        return list(ids)
    if budget <= len(omission_ids):
        return list(ids[:budget])
    remain = budget - len(omission_ids)
    head, tail = (remain + 1) // 2, remain // 2
    return list(ids[:head]) + omission_ids + (list(ids[-tail:]) if tail else [])


def allocate_budgets(lengths):
    lp, la, lb = map(int, lengths)
    available = MAX_LENGTH - overhead
    if lp + la + lb <= available:
        return lp, la, lb  # The key fix: no unnecessary prompt truncation.
    p = min(lp, PROMPT_BUDGET)
    answer_space = available - p
    a, b = min(la, answer_space // 2), min(lb, answer_space // 2)
    spare = answer_space - a - b
    # An odd spare token may favor A by one token, not an entire response.
    extra_a = min(spare, la - a)
    a += extra_a
    b += min(spare - extra_a, lb - b)
    p += min(available - p - a - b, lp - p)
    assert p + a + b <= available
    return p, a, b


def assemble_input(p, a, b):
    budgets = allocate_budgets([len(p), len(a), len(b)])
    pieces = [trim_tokens(x, k) for x, k in zip([p, a, b], budgets)]
    ids = ([tokenizer.bos_token_id] + prefix_ids + pieces[0] + a_header_ids
           + pieces[1] + b_header_ids + pieces[2] + suffix_ids + [tokenizer.eos_token_id])
    assert len(ids) <= MAX_LENGTH
    return ids, [len(x) > k for x, k in zip([p, a, b], budgets)]

In [ ]:
# Regression inputs were saved BEFORE training by your original 9B notebook.
for case in read_json(ASSET_DIR / "tokenization_cases.json"):
    parts = [tokenizer.encode(case[k], add_special_tokens=False)
             for k in ["prompt_text", "response_a_text", "response_b_text"]]
    ids, _ = assemble_input(*parts)
    if ids != case["input_ids"]:
        raise RuntimeError(f"Training/inference tokenization differs for regression case {case['id']}.")
print("Exact training-time tokenization cases passed.")


def encode_frame(frame, description):
    records = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc=description):
        p, a, b = [parse_turns(v) for v in [row.prompt, row.response_a, row.response_b]]
        n = max(map(len, [p, a, b]))
        texts = [format_turns(ts + [""] * (n-len(ts))) for ts in [p, a, b]]
        parts = [tokenizer.encode(text, add_special_tokens=False) for text in texts]
        ids, _ = assemble_input(*parts)
        records.append({"input_ids": ids, "attention_mask": [1]*len(ids)})
    return records


selection_inputs = encode_frame(selection_frame, "Tokenize selection check")
test_inputs = encode_frame(test_frame, "Tokenize test")
assert len(test_inputs) == len(test_frame)
print("Maximum encoded length:", max(len(x["input_ids"]) for x in selection_inputs + test_inputs))

## 4. Load one 9B model across the GPUs, then load checkpoint-1440

T4 ×2 uses **one model partitioned into layer groups**, not two complete model copies. Layers 0–18 and
embeddings are on GPU 0; layers 19–41 and the final head are on GPU 1. FP32 9B weights would not fit in
32 GB aggregate VRAM; FP16 is used on T4 instead. The saved files are not rewritten.

No `model.to("cuda:0")` is called after dispatch: that would defeat the two-GPU placement.
The expected missing-head message from the base checkpoint is followed by verification of the saved trained head.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select GPU T4 x2 in Kaggle Session options.")
if "model" in globals() and model is not None:
    raise RuntimeError("A model is already loaded. Restart this Kaggle session rather than load a second copy.")
ngpu = torch.cuda.device_count()
for i in range(ngpu):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | free {free/2**30:.1f}/{total/2**30:.1f} GiB")

native_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
MODEL_DTYPE = torch.bfloat16 if native_bf16 else torch.float16
cfg = read_json(BASE_DIR / "config.json")
assert cfg["model_type"] == "gemma2" and cfg["num_hidden_layers"] == 42 and cfg["hidden_size"] == 3584
if ngpu >= 2:
    USED_GPUS = [0, 1]
    if any(torch.cuda.mem_get_info(i)[0] < 12 * 2**30 for i in USED_GPUS):
        raise RuntimeError("Less than 12 GiB is free on one GPU. Restart this Kaggle session to release stale models.")
    # Disjoint modules only: a root entry plus child overrides can temporarily
    # place the ENTIRE model on GPU 0 during Accelerate hook initialization.
    device_map = {
        "model.embed_tokens": 0,
        "model.rotary_emb": 0,
        **{f"model.layers.{i}": 0 if i < 19 else 1 for i in range(42)},
        "model.norm": 1,
        "score": 1,
    }
    # Use BF16 only if EVERY device used supports it natively.
    if not all(torch.cuda.get_device_capability(i)[0] >= 8 for i in USED_GPUS):
        MODEL_DTYPE = torch.float16
elif torch.cuda.mem_get_info(0)[0] >= 21.5 * 2**30:
    USED_GPUS = [0]
    device_map = {"": 0}
else:
    raise RuntimeError("A single 16 GB GPU is insufficient for this unquantized 9B model. Select T4 x2.")
for i in USED_GPUS:
    torch.cuda.set_per_process_memory_fraction(0.90, device=i)
print("Compute dtype:", MODEL_DTYPE, "| physical inference batch:", BATCH_SIZE)

adapter_path = ADAPTER_DIR / "adapter_model.safetensors"
ac = read_json(ADAPTER_DIR / "adapter_config.json")
assert ac["task_type"] == "SEQ_CLS" and ac["peft_type"] == "LORA" and "score" in ac["modules_to_save"]
assert int(read_json(ADAPTER_DIR / "trainer_state.json")["global_step"]) == 1440
with safe_open(str(adapter_path), framework="pt", device="cpu") as f:
    saved_head = f.get_tensor(manifest["head_key"]).float()
    adapter_keys = list(f.keys())
assert tuple(saved_head.shape) == (3, 3584) and torch.isfinite(saved_head).all()

base_model = AutoModelForSequenceClassification.from_pretrained(
    str(BASE_DIR), local_files_only=True, num_labels=3, problem_type="single_label_classification",
    id2label=dict(enumerate(LABEL_NAMES)), label2id={v:i for i,v in enumerate(LABEL_NAMES)},
    pad_token_id=tokenizer.pad_token_id, dtype=MODEL_DTYPE,
    attn_implementation="eager", device_map=device_map, low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model.score.float()
# Optional incompatible TorchAO was disabled BEFORE this call in section 2.
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False,
    autocast_adapter_dtype=True, low_cpu_mem_usage=True, local_files_only=True)
model.requires_grad_(False)
model.eval()
model.config.use_cache = False
model.gradient_checkpointing_disable()
loaded_head = model.get_base_model().score.modules_to_save["default"].weight.detach().cpu().float()
torch.testing.assert_close(loaded_head, saved_head, rtol=0, atol=0)
assert sum("lora_B" in n for n,_ in model.named_parameters()) == sum("lora_B" in k for k in adapter_keys)
assert all(p.device.type == "cuda" for p in model.parameters()), "Unexpected CPU/meta offload."
assert any("lora_B" in n and torch.count_nonzero(p).item() > 0 for n,p in model.named_parameters())
INPUT_DEVICE = model.get_input_embeddings().weight.device
if len(USED_GPUS) == 2:
    assert model.get_base_model().model.layers[0].self_attn.q_proj.base_layer.weight.device.index == 0
    assert model.get_base_model().model.layers[41].self_attn.q_proj.base_layer.weight.device.index == 1
for i in USED_GPUS:
    print(f"GPU {i} model allocation: {torch.cuda.memory_allocated(i)/2**30:.2f} GiB")
del saved_head, loaded_head, base_model
# Never move the full dispatched model to one device.
gc.collect()
print("Checkpoint-1440 adapter and EXACT trained FP32 classification head verified.")

## 5. Inference helper and the full selection-set compatibility check

Dynamic right-padding, length-sorted batches, and restoring original row order preserve ID alignment.
Only a genuine CUDA out-of-memory exception triggers a smaller-batch retry. Non-finite predictions fail explicitly.
The check uses the same saved selection IDs and original A/B ordering as training evaluation.

In [ ]:
@torch.inference_mode()
def predict_logits(inputs, description):
    lengths = np.asarray([len(x["input_ids"]) for x in inputs])
    order = np.argsort(lengths, kind="stable")
    logits = np.empty((len(inputs), 3), dtype=np.float32)
    position = 0
    bs = BATCH_SIZE
    started = time.perf_counter()
    with tqdm(total=len(inputs), desc=description) as progress:
        while position < len(order):
            idx = order[position:position+bs]
            batch = out = None
            oom = False
            try:
                batch = collator([inputs[int(i)] for i in idx])
                batch = {k:v.to(INPUT_DEVICE) for k,v in batch.items()}
                with torch.autocast("cuda", dtype=MODEL_DTYPE):
                    out = model(**batch, use_cache=False).logits
                if out.shape != (len(idx), 3) or not torch.isfinite(out).all():
                    raise RuntimeError("Non-finite/invalid logits. Do not submit; share this cell's output.")
                logits[idx] = out.float().cpu().numpy()
            except torch.OutOfMemoryError:
                if bs == 1:
                    raise
                oom = True
            finally:
                del batch, out
            if oom:
                bs = max(1, bs//2)
                gc.collect()
                for device in USED_GPUS:
                    with torch.cuda.device(device):
                        torch.cuda.empty_cache()
                print("Retry with physical batch", bs)
                continue
            position += len(idx)
            progress.update(len(idx))
    return logits, time.perf_counter()-started


def probabilities_from_logits(logits):
    z = np.asarray(logits, dtype=np.float64)
    if not np.isfinite(z).all():
        raise ValueError("Non-finite logits.")
    z -= z.max(axis=1, keepdims=True)
    p = np.exp(z)
    p /= p.sum(axis=1, keepdims=True)
    assert np.isfinite(p).all() and np.allclose(p.sum(1), 1.0)
    return p


# Probe the longest REAL input first. Does not train or update weights.
all_inputs = selection_inputs + test_inputs
longest = int(np.argmax([len(x["input_ids"]) for x in all_inputs]))
_, _ = predict_logits([all_inputs[longest]], "Maximum-length smoke test")
del all_inputs
selection_logits, selection_seconds = predict_logits(selection_inputs, "Verify checkpoint-1440 selection score")
selection_probs = probabilities_from_logits(selection_logits)
selection_loss = float(-np.log(np.clip(selection_probs[np.arange(len(y_selection)), y_selection],
                                      np.finfo(np.float64).eps, 1)).mean())
expected = float(manifest["checkpoint_selection_log_loss"])
difference = selection_loss - expected
accuracy = float((selection_probs.argmax(1) == y_selection).mean())
precision_check = {"checkpoint_step": 1440, "training_selection_log_loss": expected,
    "kaggle_selection_log_loss": selection_loss, "difference": difference, "accuracy": accuracy,
    "dtype": str(MODEL_DTYPE), "selection_rows": len(selection_inputs), "seconds": selection_seconds}
(WORK_DIR / "precision_check.json").write_text(json.dumps(precision_check, indent=2))
print(json.dumps(precision_check, indent=2))
if abs(difference) > MAX_RELOAD_LOGLOSS_DIFFERENCE:
    raise RuntimeError("Selection log loss changed materially under this runtime. Share precision_check.json; do not submit yet.")

# Estimate using observed length mix. This is a warning, not a silent time/quality policy.
ratio = (np.mean([len(x["input_ids"]) for x in test_inputs]) /
         np.mean([len(x["input_ids"]) for x in selection_inputs]))
predicted_test_seconds = selection_seconds * len(test_inputs)/len(selection_inputs) * max(1.0, ratio**2)
reference_25k_hours = (selection_seconds * 25_000 / len(selection_inputs) + selection_seconds) / 3600
print(f"Approximate selection + 25,000-test-row inference cost at this speed: {reference_25k_hours:.2f} hours.")
print("This excludes loading and is not a guarantee; hidden sequence lengths and batch retries differ.")
if (time.perf_counter()-NOTEBOOK_STARTED) + 1.25*predicted_test_seconds > 8.5*3600:
    print("WARNING: measured workload may approach/exceed Kaggle's 9-hour limit. No rows will be skipped.")
del selection_inputs, selection_frame, selection_probs, selection_logits
gc.collect()

## 6. Predict every test row and write submission.csv

No swapped-order TTA, temperature fitting, or 2B blending is added to this interim checkpoint submission.
Final training may later select a different checkpoint/recipe. This result is a clean checkpoint-1440 baseline.

In [ ]:
test_logits, test_seconds = predict_logits(test_inputs, "Predict competition test")
test_probs = probabilities_from_logits(test_logits)
assert test_probs.shape == (len(test_frame), 3)
submission = pd.DataFrame(test_probs, columns=LABEL_COLUMNS)
submission.insert(0, "id", test_frame["id"].to_numpy())
# Preserve the sample submission's ID order if it contains the same runtime IDs.
sample_path = COMPETITION_DIR / "sample_submission.csv"
if sample_path.is_file():
    sample = pd.read_csv(sample_path, dtype={"id": str})
    assert list(sample.columns) == ["id", *LABEL_COLUMNS]
    if len(sample) == len(submission) and set(sample["id"]) == set(submission["id"]):
        submission = submission.set_index("id").loc[sample["id"]].reset_index()
assert len(submission) == len(test_frame) and submission["id"].is_unique
assert set(submission["id"]) == set(test_frame["id"])
values = submission[LABEL_COLUMNS].to_numpy()
assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
assert np.allclose(values.sum(1), 1.0, atol=1e-8)
output_path = WORK_DIR / "submission.csv"
submission.to_csv(output_path, index=False)
readback = pd.read_csv(output_path, dtype={"id": str})
assert readback["id"].equals(submission["id"])
assert list(readback.columns) == ["id", *LABEL_COLUMNS]
assert np.allclose(readback[LABEL_COLUMNS].sum(axis=1), 1.0, atol=1e-8)
report = {"checkpoint_step":1440, "base_revision":manifest["base_revision"],
    "adapter_sha256":manifest["adapter_sha256"], "rows":len(submission), "max_length":MAX_LENGTH,
    "dtype":str(MODEL_DTYPE), "devices":[torch.cuda.get_device_name(i) for i in USED_GPUS],
    "precision_check":precision_check, "test_seconds":test_seconds,
    "total_seconds":time.perf_counter()-NOTEBOOK_STARTED,
    "swap_tta":False, "temperature":1.0, "ensemble":False,
    "versions":{n:metadata.version(n) for n in ["torch","transformers","peft","accelerate"]}}
(WORK_DIR / "inference_report.json").write_text(json.dumps(report, indent=2))
print(submission.head().to_string(index=False))
print(f"\nWrote {len(submission):,} rows to {output_path}")
print("When Save & Run All succeeds, submit this SAVED VERSION's submission.csv to the competition.")
print("Kaggle re-runs it on the hidden test; a three-row development CSV is not the hidden-test result.")

## References and test scope

- [Official Gemma 2 integration/precision guidance](https://huggingface.co/blog/gemma2): FP16 is an inference option for the 9B model; this does not guarantee identical fine-tuned probabilities.
- [Accelerate device-map inference](https://huggingface.co/docs/accelerate/usage_guides/big_modeling): layer placement across devices. This is model parallelism, not DDP or a 2x throughput guarantee.
- [PEFT checkpoint format](https://huggingface.co/docs/peft/developer_guides/checkpoint): base weights and adapters are separate.
- [Competition code requirements](https://www.kaggle.com/competitions/llm-classification-finetuning): offline, nine-hour execution limit, output named `submission.csv`.

Prepared from your uploaded 9B notebook and exact preprocessing functions. Static checks and synthetic CPU tests were performed.
**Actual 9B/T4×2 loading, FP16 score parity, and hidden-test runtime were not tested in the assistant environment.**
This notebook performs the full selection compatibility check on your Kaggle run before producing the submission.